# 🎭 Output Parsers in LangChain

## Learning Objectives
In this notebook, you will learn:
1. **StrOutputParser** - How to extract plain-text responses from a chain
2. **JsonOutputParser** - How to coerce an LLM response into a raw JSON object
3. **PydanticOutputParser** - How to parse structured LLM output into a validated Pydantic model
4. **`with_structured_output`** - How to bind a schema directly to a model for structured generation

## Prerequisites
- Basic familiarity with LangChain prompts and chains (LCEL `|` syntax)
- Familiarity with Pydantic `BaseModel` and `Field`
- An `OPENAI_API_KEY` set in a `.env` file at the project root


---
## 🔧 Setup: Environment & LLM Initialization

We load environment variables from `.env` and initialize the chat model used throughout this notebook. All four output-parser strategies below reuse this same `llm` instance.

In [ ]:
# ============================================================================
# SETUP: Imports & LLM Initialization
# ============================================================================
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_openai import ChatOpenAI

load_dotenv()

llm = init_chat_model(model="gpt-4o-mini", temperature=0)

print(f"🤖 LLM initialized: {llm.model_name if hasattr(llm, 'model_name') else llm}")
print("✅ Environment ready!")

---
## 📝 Part 1: `StrOutputParser`

The simplest output parser. It takes the raw LLM response (an `AIMessage`) and returns just the `content` string, which is handy when you don't need any further structure.

In [ ]:
# ============================================================================
# STR OUTPUT PARSER: Plain-Text Chain
# ============================================================================
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("Write a short poem about {topic}")

chain = prompt | llm | parser

response = chain.invoke({"topic": "nature"})

print("📄 Response type:", type(response))
print(response)

---
## 📦 Part 2: `JsonOutputParser`

`JsonOutputParser` asks the model to return a JSON blob and parses it into a plain Python `dict`. Useful for quick structured extraction without defining a full schema.

In [ ]:
# ============================================================================
# JSON OUTPUT PARSER: Dict-Based Chain
# ============================================================================
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_template(
    "Return a JSON object with 'name' and 'age' for: {description}"
)

chain = prompt | llm | parser

result = chain.invoke({"description": "A 25-year-old developer named Alex"})

print("📦 Parsed JSON:", result)  # {'name': 'Alex', 'age': 25}

---
## 🔒 Part 3: `PydanticOutputParser`

For validated, typed output, `PydanticOutputParser` binds the response to a Pydantic model. The parser injects format instructions into the prompt (via `.partial()`) so the model knows the exact schema to produce, and returns a validated model instance instead of a raw dict.

### `Person`

In [ ]:
# ============================================================================
# PYDANTIC OUTPUT PARSER: `Person` Schema
# ============================================================================
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class Person(BaseModel):
    name: str = Field(description="The person's name")
    age: int = Field(description="The person's age")
    occupation: str = Field(description="The person's occupation")


parser = PydanticOutputParser(pydantic_object=Person)

In [ ]:
# ============================================================================
# PYDANTIC OUTPUT PARSER: Build & Run the Chain
# ============================================================================
prompt = ChatPromptTemplate.from_template(
    "Return a JSON object with 'name', 'age', and 'occupation' for: {description}"
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({"description": "A 30-year-old artist named Maria"})

print(result)  # Person(name='Maria', age=30, occupation='artist')

---
## 🎯 Part 4: `with_structured_output`

Rather than parsing text after the fact, `llm.with_structured_output(Schema)` binds the schema directly to the model call. The model is instructed (and, for supporting providers, constrained) to return data matching the schema, so you get a validated Pydantic instance back with no separate parser step.

### `MovieReview`

In [ ]:
# ============================================================================
# STRUCTURED OUTPUT: `MovieReview` Schema
# ============================================================================
class MovieReview(BaseModel):
    title: str = Field(description="The title of the movie")
    review: str = Field(description="A brief review of the movie")
    rating: int = Field(description="The rating of the movie out of 10")


# Bind the schema directly to the model
structured_model = llm.with_structured_output(MovieReview)

In [ ]:
# ============================================================================
# STRUCTURED OUTPUT: Run the Model
# ============================================================================
result = structured_model.invoke("Review: Inception is a mind-bending thriller. 9/10")

print(result)  # MovieReview(title='Inception', review='A mind-bending thriller.', rating=9)

---
## 📝 Summary

In this notebook, we explored four ways to shape LLM output in LangChain:

### 1. Text & JSON Parsers
- **`StrOutputParser`**: Returns the raw text content of the LLM response
- **`JsonOutputParser`**: Parses the response into a plain Python `dict`

### 2. Schema-Validated Parsers
- **`PydanticOutputParser`**: Injects format instructions into the prompt and validates the response against a `Person` model
- **`with_structured_output`**: Binds a `MovieReview` schema directly to the model call for validated structured generation, no separate parser needed

### Next Steps
- Compare these approaches against tool-calling-based structured extraction
- Move on to the next notebook to see agents and tool-calling in action
